<div style="background-color: #f0f2f5; padding: 20px; border-radius: 10px; border-left: 10px solid #20beff;">
    <h1 style="color: #1c1c1c; margin-bottom: 5px;">The Simplest CatBoost Guide🐈: Predicting Student Test Score</h1>
    <p style="color: #666;">By Kaggle Contrubutor</p>
</div>

Im just a Contributor, but created this to understand and review the behavior of CatBoostand Target Encoding. If you have an advice, Im happy if you leave a comment(still for my Engkish Skill)

In [2]:
# !pip install pandas numpy seaborn scikit-learn catboost

In [3]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from catboost import CatBoostRegressor

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
# ----------------------------------------------------------------
# Utility Functions
# ----------------------------------------------------------------
def normalize_columns(df):
    """
    Standardizes column names by replacing spaces with underscores, 
    converting to lowercase, and removing duplicates.
    """
    df.columns = [c.replace(' ', '_').strip().lower() for c in df.columns] #Replace spaces, strip whitespace, and convert to lowercase
    return df.loc[:, ~df.columns.duplicated()] #Drop duplicated columns while keeping the first occurrence

# ----------------------------------------------------------------
# Data Loading
# ----------------------------------------------------------------
# Note: Ensure the paths match your Kaggle input directory
trn_df = pd.read_csv("playground-series-s6e1/train.csv") #to combine original data later, now naming it as 'trn'
test_df = pd.read_csv("playground-series-s6e1/test.csv")
orig_df = pd.read_csv("playground-series-s6e1/StudentPerformanceFactors.csv") #competetion's original dataset

# Normalize all dataframes
trn_df = normalize_columns(trn_df)
test_df = normalize_columns(test_df)
orig_df = normalize_columns(orig_df)

# ----------------------------------------------------------------
# External Dataset Alignment
# ----------------------------------------------------------------
# Align column names of the original dataset with the competition data
rename_map = {
    'hours_studied': 'study_hours',
    'attendance': 'class_attendance',
}
orig_df = orig_df.rename(columns=rename_map) # bulk transformation using mapping

# Standardize categorical values (fix casing inconsistencies between datasets)
cat_cols = ['gender', 'internet_access']
for col in cat_cols:
    for df in [trn_df, test_df, orig_df]:
        if col in df.columns:
            df[col] = df[col].astype(str).str.lower()

# ----------------------------------------------------------------
# Ordinal Encoding
# ----------------------------------------------------------------
# Mapping ordinal categories to numerical values for better interpretability
ordinal_maps = {
    "sleep_quality": {"poor": 0, "average": 1, "good": 2},
    "facility_rating": {"low": 0, "medium": 1, "high": 2},
    "exam_difficulty": {"easy": 0, "moderate": 1, "hard": 2},
    "internet_access": {"no": 0, "yes": 1}
}

for df in [trn_df, test_df, orig_df]:
    for col, mapping in ordinal_maps.items():
        if col in df.columns:
            # Create a new numerical column while retaining the original
            df[f"{col.split('_')[0]}_num"] = df[col].map(mapping)

# ----------------------------------------------------------------
# Data Integration
# ----------------------------------------------------------------
# Add source flags to distinguish between competition and external data
trn_df['is_original'] = 0
orig_df['is_original'] = 1
test_df['is_original'] = 0 #Placeholder for test set alignment

# Merge using only common columns found in both datasets
common_cols = list(set(trn_df.columns) & set(orig_df.columns))
train_df = pd.concat([trn_df, orig_df[common_cols]], axis=0, ignore_index=True) #axis=0は、行方向（縦方向）に繋げるという意味

seed = 42 #Standard random seed for reproducibility(wordplay?)

print(f"Combined Train Shape: {train_df.shape}")

FileNotFoundError: [Errno 2] No such file or directory: 'KaggleData/playground-series-s6e1/train.csv'

In [ ]:
# ----------------------------------------------------------------
# Feature Engineering
# ----------------------------------------------------------------
def make_features(df):
    """
    Creates new features based on domain knowledge and economic intuition.
    """
    # High Study Flag: Binary indicator for intensive study (7+ hours)
    df['high_study'] = (df['study_hours'] >= 7).astype(int)
    
    # Quadratic Terms: Capturing Diminishing Marginal Utility
    # While linear terms assume a constant effect, polynomial terms allow the model 
    # to capture non-linear relationships, such as the diminishing returns of study time.
    df['study_hours_sq'] = df['study_hours'] ** 2
    df['class_attendance_sq'] = df['class_attendance'] ** 2
    
    # Time Resource Allocation
    # Although sleep_hours and active_hours are highly collinear, 
    # we rely on CatBoost's robustness to handle these relationships.
    df['active_hours'] = 24 - df["sleep_hours"] 
    df['free_time'] = df['active_hours'] - df['study_hours']       
    
    return df
    
# Apply feature generation
train_df = make_features(train_df)
test_df = make_features(test_df)

# ----------------------------------------------------------------
# Target Encoding (with Smoothing and Cross-Validation)
# ----------------------------------------------------------------
# To prevent Data Leakage, we implement Target Encoding with a 
# combination of smoothing and K-Fold cross-validation.

def target_encode_smooth(df, col, target, m=20):
    """
    Smooths categorical target means to prevent overfitting on categories with small sample sizes.
    Formula: (category_mean * count + global_mean * m) / (count + m)
    A higher 'm' (smoothing factor) pulls category means closer to the global average.
    """
    global_mean = df[target].mean() # Overall population mean
    stats = df.groupby(col)[target].agg(['mean', 'count'])

    # Applying the smoothing formula
    smooth = (stats['count'] * stats['mean'] + m * global_mean) / (stats['count'] + m)
    return smooth

# Cross-Validation for Target Encoding to strictly avoid leakage.
# Encoding for each fold is calculated using data from the other folds only.
kf = KFold(n_splits=5, shuffle=True, random_state=seed)
overall_mean = train_df['exam_score'].mean() # Default mean for imputation

for col in ["course", "study_method"]:
    new_col = f"{col}_te"
    train_df[new_col] = 0

    # Iterative encoding via K-Fold splits
    for tr_idx, val_idx in kf.split(train_df):
        # Calculate smoothing map using training folds only
        smooth_map = target_encode_smooth(
            train_df.iloc[tr_idx], col, "exam_score", m=20
        )
        # Apply the map to the validation fold
        train_df.loc[val_idx, new_col] = train_df.loc[val_idx, col].map(smooth_map)
        
    # Impute any missing values with the overall global mean
    train_df[new_col] = train_df[new_col].fillna(overall_mean)

    # For the test set, use the mapping derived from the entire training dataset
    smooth_map = target_encode_smooth(train_df, col, "exam_score", m=20)
    test_df[new_col] = test_df[col].map(smooth_map).fillna(overall_mean)

In [ ]:
# ----------------------------------------------------------------
# Feature Selection and Preparation
# ----------------------------------------------------------------
# Define columns to drop (identifiers and raw categorical columns replaced by TE)
drop_cols = ["id", "age", "study_method", "course"]

X = train_df.drop(["exam_score"] + [c for c in drop_cols if c in train_df.columns], axis=1)
X_test = test_df.drop([c for c in drop_cols if c in test_df.columns] + ["id"], axis=1)
y = train_df["exam_score"]

# CRITICAL: Ensure the column order in the test set perfectly matches the training set.
# This is vital when merging external datasets or performing extensive feature engineering.
X_test = X_test[X.columns]

# List of categorical features for CatBoost
cat_features = ["gender", "internet_access", "sleep_quality", "facility_rating", "exam_difficulty"]

# Convert categorical features to string type (CatBoost requirement)
# Even if they appear as strings, pandas may store them as objects. 
# Explicit conversion ensures the model processes them correctly as categories.
for col in cat_features:
    X[col] = X[col].astype(str)
    X_test[col] = X_test[col].astype(str)

In [ ]:
# ----------------------------------------------------------------
# Model Training (Cross-Validation)
# ----------------------------------------------------------------
# Initialize arrays to store Out-of-Fold (OOF) predictions and test set predictions.
# OOF predictions represent the model's performance on "unseen" data during training.
oof = np.zeros(len(X)) 
test_preds = np.zeros(len(X_test)) 

for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
    print(f"--- Processing Fold {fold+1} ---")

    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    # Model Configuration: Utilizing CatBoost for its superior handling of categorical data.
    model = CatBoostRegressor(
        iterations=8000, 
        learning_rate=0.03,
        depth=5,
        loss_function="RMSE",
        random_seed=seed,
        subsample=0.8,
        verbose=400,            # Output logs every 400 iterations
        early_stopping_rounds=200 # Stop training if the validation score stops improving
    )

    # Execute Training
    model.fit(
        X_tr, y_tr,
        eval_set=(X_val, y_val), # Use validation set for early stopping
        cat_features=cat_features,
        use_best_model=True
    )

    # Generate Predictions
    # Save validation fold predictions to the OOF array
    oof[val_idx] = model.predict(X_val)
    # Average test set predictions across all folds (Ensembling)
    test_preds += model.predict(X_test) / kf.n_splits

# ----------------------------------------------------------------
# Final Evaluation
# ----------------------------------------------------------------
# Calculate the overall Root Mean Squared Error (RMSE) based on OOF predictions.
rmse = np.sqrt(mean_squared_error(y, oof))
print(f"\nFinal OOF RMSE: {rmse:.4f}")

--- Fold 1 ---
0:	learn: 18.4862624	test: 18.4483320	best: 18.4483320 (0)	total: 109ms	remaining: 14m 32s
400:	learn: 8.8009142	test: 8.7993088	best: 8.7993088 (400)	total: 18s	remaining: 5m 41s
800:	learn: 8.7723075	test: 8.7746422	best: 8.7746422 (800)	total: 36s	remaining: 5m 23s
1200:	learn: 8.7535114	test: 8.7599566	best: 8.7599566 (1200)	total: 54.4s	remaining: 5m 7s
1600:	learn: 8.7391732	test: 8.7500734	best: 8.7500734 (1600)	total: 1m 12s	remaining: 4m 51s
2000:	learn: 8.7270514	test: 8.7425076	best: 8.7425076 (2000)	total: 1m 31s	remaining: 4m 35s
2400:	learn: 8.7164821	test: 8.7365226	best: 8.7365226 (2400)	total: 1m 50s	remaining: 4m 18s
2800:	learn: 8.7068776	test: 8.7312152	best: 8.7312152 (2800)	total: 2m 11s	remaining: 4m 3s
3200:	learn: 8.6983713	test: 8.7274331	best: 8.7274331 (3200)	total: 2m 30s	remaining: 3m 45s
3600:	learn: 8.6904088	test: 8.7238277	best: 8.7238277 (3600)	total: 2m 49s	remaining: 3m 26s
4000:	learn: 8.6833719	test: 8.7213053	best: 8.7213053 (4000)

# 🔱 Student Test Score Prediction: CatBoost & Economic Feature Engineering
Created for educational purposes to understand Target Encoding and CatBoost's robust handling of categorical data.

### 🔱 Strategy: Why CatBoost?
1. **Ordered Target Statistics**: 
Prevents data leakage by using a sequential encoding process on a virtual time axis (random permutations). This ensures robust generalization and minimizes the gap between CV and Leaderboard.
2. **Feature Combinations**: 
Automatically captures synergies between categorical variables (e.g., `Course` × `Study Method`) without manual interaction term generation.
3. **Symmetric Trees**: 
Ensures high regularization by forcing identical splits across the same tree level, which physically suppresses overfitting (variance).

### 🔱 Feature Engineering: The Economic Lens
* **Diminishing Returns**: Added `study_hours_sq` and `attendance_sq` to capture the non-linear "Marginal Utility" of study time.
* **Resource Allocation**: Derived `free_time` from sleep and study hours to represent the student's time budget constraint.

In [ ]:
# ----------------------------------------------------------------
# Feature Importance Analysis
# ----------------------------------------------------------------
# Extracting feature importance to understand the model's decision-making process.
feat_imp = model.get_feature_importance(prettified=True)

# Visualize the Top 10 most influential features
plt.figure(figsize=(10, 6))
sns.barplot(x="Importances", y="Feature Id", data=feat_imp.head(10), palette="viridis")
plt.title("CatBoost Feature Importance: Top 10 Contributors", fontsize=14)
plt.xlabel("Importance Score")
plt.ylabel("Features")
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

# ----------------------------------------------------------------
# Final Submission File Generation
# ----------------------------------------------------------------
submission = pd.DataFrame({
    "id": test_df["id"],
    "exam_score": test_preds
})
submission.to_csv("submission.csv", index=False)
print("Success: Submission file 'submission.csv' has been created!")

NameError: name 'model' is not defined

In [ ]:
submission = pd.DataFrame({
    "id": test_df["id"],
    "exam_score": test_preds
})
submission.to_csv("submission.csv", index=False)
print("Submission file created!")

Submission file created!
